# 02 Model Development

**Project:** Heart Disease MLOps  
**Purpose:** Train, tune, compare, evaluate, and package classification models using the cleaned Cleveland heart-disease dataset.

This notebook expects Notebook 1 to have created:

`HeartDisease_MLOps/data/processed/cleaned_heart_disease.csv`

Run the cells sequentially from top to bottom.

## 1. Environment setup

This cell installs the required libraries in Google Colab. The quiet option keeps the output concise.

In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib

## 2. Imports and reproducibility settings

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

print("Imports completed successfully.")

## 3. Project paths

The notebook searches common Colab locations for the project folder. It also creates output directories if needed.

In [ ]:
candidate_roots = [
    Path("/content/HeartDisease_MLOps"),
    Path("HeartDisease_MLOps"),
    Path("/content/drive/MyDrive/HeartDisease_MLOps"),
]

PROJECT_ROOT = next((p for p in candidate_roots if p.exists()), candidate_roots[0])
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_heart_disease.csv"
MODELS_DIR = PROJECT_ROOT / "models"
SCREENSHOTS_DIR = PROJECT_ROOT / "screenshots"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [MODELS_DIR, SCREENSHOTS_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

> If `Dataset exists` is `False`, upload the project folder to Colab or mount Google Drive, then update `PROJECT_ROOT`.

## 4. Load the processed dataset

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Processed dataset not found at {DATA_PATH}. "
        "Run Notebook 1 first or correct PROJECT_ROOT."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())

## 5. Validate the modeling dataset

These checks prevent silent failures caused by duplicate rows, unexpected target values, missing columns, or non-numeric data.

In [ ]:
EXPECTED_COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

# Coerce all expected fields to numeric and report any conversion problems.
for column in EXPECTED_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))
print("Duplicate rows:", df.duplicated().sum())
print("Target values:", sorted(df["target"].dropna().unique().tolist()))

if df["target"].isna().any():
    raise ValueError("Target contains missing values.")
if not set(df["target"].unique()).issubset({0, 1}):
    raise ValueError("Target must be binary with values 0 and 1.")

# Remove duplicate rows only. Feature missing values, if any, are handled by the pipeline.
df = df.drop_duplicates().reset_index(drop=True)
print("Validated dataset shape:", df.shape)

## 6. Define features and target

Categorical-looking integer fields are one-hot encoded. Continuous fields are median-imputed and standardized. This preprocessing is packaged inside each model pipeline, preventing train-test leakage.

In [ ]:
X = df.drop(columns="target")
y = df["target"].astype(int)

continuous_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Continuous features:", continuous_features)
print("Categorical features:", categorical_features)
display(y.value_counts().rename_axis("target").to_frame("count"))

## 7. Stratified train-test split

The untouched test set is reserved for final evaluation. Stratification preserves the target proportion in both subsets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)
print("Training target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())
print("Test target distribution:")
display(y_test.value_counts(normalize=True).rename("proportion").to_frame())

## 8. Build reusable preprocessing and model pipelines

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("continuous", numeric_pipeline, continuous_features),
    ("categorical", categorical_pipeline, categorical_features),
])

logistic_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

random_forest_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE)),
])

models = {
    "Logistic Regression": logistic_pipeline,
    "Random Forest": random_forest_pipeline,
}

print("Pipelines created successfully.")

## 9. Five-fold stratified cross-validation

The notebook compares both baseline models using accuracy, precision, recall, F1, and ROC-AUC.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_rows = []
for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    row = {"model": model_name}
    for metric in scoring:
        row[f"cv_{metric}_mean"] = scores[f"test_{metric}"].mean()
        row[f"cv_{metric}_std"] = scores[f"test_{metric}"].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values("cv_roc_auc_mean", ascending=False)
display(cv_results.round(4))
cv_results.to_csv(REPORTS_DIR / "baseline_cross_validation_results.csv", index=False)
print("Saved:", REPORTS_DIR / "baseline_cross_validation_results.csv")

## 10. Hyperparameter tuning

`GridSearchCV` searches model-specific parameter combinations using mean cross-validated ROC-AUC as the selection criterion.

In [ ]:
parameter_grids = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0, 100.0],
        "classifier__solver": ["liblinear", "lbfgs"],
        "classifier__class_weight": [None, "balanced"],
    },
    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 4, 8],
        "classifier__min_samples_split": [2, 5],
        "classifier__min_samples_leaf": [1, 2],
        "classifier__class_weight": [None, "balanced"],
    },
}

searches = {}
tuning_rows = []

for model_name, model in models.items():
    print(f"Tuning {model_name}...")
    search = GridSearchCV(
        estimator=model,
        param_grid=parameter_grids[model_name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[model_name] = search
    tuning_rows.append({
        "model": model_name,
        "best_cv_roc_auc": search.best_score_,
        "best_parameters": json.dumps(search.best_params_),
    })
    print("Best CV ROC-AUC:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)
    print()

tuning_results = pd.DataFrame(tuning_rows).sort_values("best_cv_roc_auc", ascending=False)
display(tuning_results)
tuning_results.to_csv(REPORTS_DIR / "hyperparameter_tuning_results.csv", index=False)
print("Saved:", REPORTS_DIR / "hyperparameter_tuning_results.csv")

## 11. Evaluate tuned models on the untouched test set

ROC-AUC is calculated from predicted probabilities, not from binary predictions.

In [ ]:
evaluation_rows = []
prediction_store = {}

for model_name, search in searches.items():
    fitted_model = search.best_estimator_
    predictions = fitted_model.predict(X_test)
    probabilities = fitted_model.predict_proba(X_test)[:, 1]

    prediction_store[model_name] = {
        "predictions": predictions,
        "probabilities": probabilities,
    }

    evaluation_rows.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities),
    })

test_results = pd.DataFrame(evaluation_rows).sort_values("roc_auc", ascending=False)
display(test_results.round(4))
test_results.to_csv(REPORTS_DIR / "test_set_model_comparison.csv", index=False)
print("Saved:", REPORTS_DIR / "test_set_model_comparison.csv")

## 12. Classification reports

In [ ]:
for model_name, stored in prediction_store.items():
    print("=" * 70)
    print(model_name)
    print(classification_report(
        y_test,
        stored["predictions"],
        target_names=["No disease", "Disease"],
        digits=4,
        zero_division=0,
    ))

## 13. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, len(searches), figsize=(12, 5))
if len(searches) == 1:
    axes = [axes]

for ax, (model_name, stored) in zip(axes, prediction_store.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        stored["predictions"],
        display_labels=["No disease", "Disease"],
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(model_name)

fig.suptitle("Confusion Matrices for Tuned Models", fontsize=14)
plt.tight_layout()
confusion_path = SCREENSHOTS_DIR / "model_confusion_matrices.png"
plt.savefig(confusion_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", confusion_path)

## 14. ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for model_name, stored in prediction_store.items():
    RocCurveDisplay.from_predictions(
        y_test,
        stored["probabilities"],
        name=model_name,
        ax=ax,
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
ax.set_title("ROC Curves for Tuned Models")
ax.legend(loc="lower right")
plt.tight_layout()
roc_path = SCREENSHOTS_DIR / "model_roc_curves.png"
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", roc_path)

## 15. Select and save the final model

The final model is selected using cross-validated ROC-AUC from the training data, avoiding test-set-driven model selection. The saved artifact contains preprocessing and classification in one pipeline.

In [ ]:
best_model_name = tuning_results.iloc[0]["model"]
best_model = searches[best_model_name].best_estimator_
best_cv_score = searches[best_model_name].best_score_

model_path = MODELS_DIR / "heart_disease_pipeline.joblib"
joblib.dump(best_model, model_path)

metadata = {
    "model_name": best_model_name,
    "selection_metric": "mean 5-fold stratified CV ROC-AUC on training data",
    "selection_score": float(best_cv_score),
    "best_parameters": searches[best_model_name].best_params_,
    "input_features": X.columns.tolist(),
    "continuous_features": continuous_features,
    "categorical_features": categorical_features,
    "target_mapping": {"0": "No disease", "1": "Disease"},
    "random_state": RANDOM_STATE,
}
metadata_path = MODELS_DIR / "heart_disease_pipeline_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Selected model:", best_model_name)
print("Selection CV ROC-AUC:", round(best_cv_score, 4))
print("Saved model:", model_path)
print("Saved metadata:", metadata_path)

## 16. Model reload and prediction smoke test

This confirms that the serialized pipeline can be loaded independently and can return both a class and confidence value.

In [ ]:
loaded_model = joblib.load(model_path)
sample_input = X_test.iloc[[0]].copy()
sample_prediction = int(loaded_model.predict(sample_input)[0])
sample_probability = float(loaded_model.predict_proba(sample_input)[0, 1])

print("Sample input:")
display(sample_input)
print("Predicted class:", sample_prediction)
print("Predicted label:", "Disease" if sample_prediction == 1 else "No disease")
print("Probability of disease:", round(sample_probability, 4))

## 17. Permutation feature importance for the selected pipeline

Permutation importance measures the decrease in test-set ROC-AUC when each original input feature is shuffled. It works consistently for either selected model.

In [ ]:
importance = permutation_importance(
    loaded_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)

display(importance_df.round(4))
importance_df.to_csv(REPORTS_DIR / "permutation_feature_importance.csv", index=False)

plt.figure(figsize=(9, 6))
sns.barplot(
    data=importance_df,
    x="importance_mean",
    y="feature",
    color="steelblue",
)
plt.xlabel("Mean decrease in ROC-AUC after permutation")
plt.ylabel("Input feature")
plt.title(f"Permutation Feature Importance: {best_model_name}")
plt.tight_layout()
importance_path = SCREENSHOTS_DIR / "permutation_feature_importance.png"
plt.savefig(importance_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", importance_path)

## 18. Final artifact checklist

In [ ]:
artifacts = [
    DATA_PATH,
    REPORTS_DIR / "baseline_cross_validation_results.csv",
    REPORTS_DIR / "hyperparameter_tuning_results.csv",
    REPORTS_DIR / "test_set_model_comparison.csv",
    REPORTS_DIR / "permutation_feature_importance.csv",
    SCREENSHOTS_DIR / "model_confusion_matrices.png",
    SCREENSHOTS_DIR / "model_roc_curves.png",
    SCREENSHOTS_DIR / "permutation_feature_importance.png",
    MODELS_DIR / "heart_disease_pipeline.joblib",
    MODELS_DIR / "heart_disease_pipeline_metadata.json",
]

artifact_check = pd.DataFrame({
    "artifact": [str(path) for path in artifacts],
    "exists": [path.exists() for path in artifacts],
})
display(artifact_check)

if artifact_check["exists"].all():
    print("All expected model-development artifacts are available.")
else:
    print("One or more artifacts are missing. Review the failed or skipped cells above.")

## Conclusion

This notebook:

- loaded and validated the processed heart-disease dataset;
- applied leakage-safe preprocessing inside scikit-learn pipelines;
- compared Logistic Regression and Random Forest with stratified cross-validation;
- tuned both models with `GridSearchCV`;
- evaluated tuned models using accuracy, precision, recall, F1, ROC-AUC, confusion matrices, and ROC curves;
- selected the final model using training cross-validation performance;
- saved one deployable preprocessing-plus-model pipeline and supporting metadata;
- verified serialized inference and generated permutation feature importance.

The next notebook should use MLflow to log parameters, metrics, plots, and model artifacts for these experiments.